In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Mnist CNN model

In [35]:
#GPU 연결
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [36]:
#dataset 정의 및 로더
batch_size = 50
mnist_train = datasets.MNIST(root='MNIST_data/',
                          train=True,
                          transform=transforms.ToTensor(),
                          download=True)

mnist_test = datasets.MNIST(root='MNIST_data/',
                         train=False,
                         transform=transforms.ToTensor(),
                         download=True)

train_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                          shuffle=True,
                                          drop_last=True)

test_loader = torch.utils.data.DataLoader(dataset=mnist_test,
                                          batch_size=1000,
                                          shuffle=False)

In [37]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.fc = nn.Linear(7 * 7 * 64, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = torch.flatten(x,1)
        x = self.fc(x)
        return x

mnist_model = CNN().to(device)

criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(mnist_model.parameters(), lr=0.001)

In [38]:
epochs = 5
total_batch = len(train_loader)

mnist_model.train()
for epoch in range(epochs):
    avg_cost = 0

    for X, Y in train_loader:
        X = X.to(device) 
        Y = Y.to(device) 

        optimizer.zero_grad()
        output = mnist_model(X)
        cost = criterion(output, Y)
        cost.backward()
        optimizer.step()
        
        avg_cost += cost / total_batch

    print('[Epoch: {}] cost = {:.6f}'.format(epoch + 1, avg_cost))

torch.save(mnist_model.state_dict(), "my_model.pth")
print("모델 저장 완료")

[Epoch: 1] cost = 0.161197
[Epoch: 2] cost = 0.051354
[Epoch: 3] cost = 0.037788
[Epoch: 4] cost = 0.028810
[Epoch: 5] cost = 0.024449
모델 저장 완료


In [39]:
print("Evaluating Model...")
mnist_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X, Y in test_loader:
        X = X.to(device)
        Y = Y.to(device)

        prediction = mnist_model(X)
        _, predicted_classes = torch.max(prediction.data, 1)
        total += Y.size(0)
        correct += (predicted_classes == Y).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy on MNIST: {accuracy:.2f}%')

Evaluating Model...
Test Accuracy on MNIST: 98.88%


# cifar10 pretrained model

In [40]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [41]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

cifar_test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
cifar_test_loader = DataLoader(cifar_test_set, batch_size=128, shuffle=False)

Files already downloaded and verified


In [42]:
class NormalizationLayer(nn.Module):
    def __init__(self):
        super(NormalizationLayer, self).__init__()
        self.mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
        self.std = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)

    def forward(self, x):
        return (x - self.mean.to(x.device)) / self.std.to(x.device)

# 출처: https://github.com/chenyaofo/pytorch-cifar-models
raw_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True)
cifar_model = nn.Sequential(
    NormalizationLayer(),
    raw_model
).to(device)

cifar_model.eval()

Using cache found in C:\Users\USER/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


Sequential(
  (0): NormalizationLayer()
  (1): CifarResNet(
    (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        

In [49]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in cifar_test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cifar_model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy on CIFAR-10: {accuracy:.2f}%')

Test Accuracy on CIFAR-10: 92.60%
